# Eksperimen pengenalan vowel: DTW vs Gaussian

Notebook ini memindai dataset WAV secara rekursif, membangun model dari `data/template`, lalu mengevaluasi rekaman template, closed, dan open. Semua prediksi adalah **kelas vowel**; folder speaker pada template/closed dipakai sebagai identitas untuk analisis per-speaker, sedangkan nama folder vowel pada open dipakai sebagai label aktual.

Jalankan notebook dari mana saja di dalam repositori. Bila root proyek tidak terdeteksi otomatis, isi `PROJECT_ROOT` pada sel konfigurasi.


## Struktur data yang didukung

```text
data/template/<speaker>/*.wav       # train dan closed recordings
data/test/closed/<speaker>/*.wav    # closed speakers
data/test/open/<vowel>/*.wav       # open; folder A/E/I/O/U adalah label vowel
```

Nama file juga dibaca dengan toleran, termasuk `S01_a.wav`, `G09_S01_A01.wav`, dan `S2_Closed_i.wav`. Untuk `open`, folder vowel adalah sumber label utama; suffix nama file menjadi fallback dan dipakai untuk validasi. Untuk template/closed, speaker utama berasal dari folder speaker, bukan dari ID rekaman di nama file.


## Metode

- Fitur: MFCC 39 dimensi (13 MFCC + delta + delta-delta), frame 25 ms dan langkah 10 ms, sesuai `features.py`.
- DTW: jarak Euclidean per frame, tiga langkah standar, biaya dibagi panjang alignment; template dibangun dengan alignment ke rekaman berpanjang median (`dtw.py`).
- Gaussian: satu Gaussian full-covariance per vowel dari seluruh frame latih; prediksi memakai negative log-likelihood rata-rata (`gaussian.py`).
- Akurasi dihitung dari label vowel aktual vs prediksi. Rata-rata per skenario di tabel speaker adalah macro-average speaker; ringkasan utama adalah macro-average skenario yang tersedia.

Audio stereo dikonversi ke mono dan sample rate selain 16 kHz di-resample sebelum MFCC. Prediksi open mengukur pengenalan vowel pada speaker yang tidak menjadi identitas folder template, bukan pengenalan identitas speaker.


In [ ]:
from pathlib import Path
from collections import Counter
from math import gcd
import csv
import re
import sys

import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
from IPython.display import display, Markdown

# Opsional: Path(r"C:/path/ke/repositori"). Jika None, notebook mencari root dari cwd.
PROJECT_ROOT = None
TARGET_SR = 16000

def find_project_root(start=Path.cwd()):
    if PROJECT_ROOT is not None:
        root = Path(PROJECT_ROOT).expanduser().resolve()
        if not (root / "data" / "template").is_dir():
            raise FileNotFoundError(f"Folder data/template tidak ditemukan di {root}")
        return root
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data" / "template").is_dir() and (candidate / "preprocessing.py").is_file():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan. Atur PROJECT_ROOT di sel konfigurasi.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from features import extract_mfcc
from preprocessing import VOWELS
from dtw import train_dtw, predict_dtw
from gaussian import train_gaussian, predict_gaussian
VOWELS = tuple(v.lower() for v in VOWELS)

SCENARIOS = {
    "Closed Recording": ROOT / "data" / "template",
    "Closed Speakers": ROOT / "data" / "test" / "closed",
    "Open Speakers": ROOT / "data" / "test" / "open",
}
print(f"Root proyek: {ROOT}\nPython: {sys.version.split()[0]}")


In [ ]:
# Label vowel yang disematkan di akhir nama file. Mendukung suffix seperti A01.
VOWEL_SUFFIX = re.compile(r"(?:^|[_-])([aiueo])(?:\d+)?$", re.IGNORECASE)

def vowel_from_filename(path):
    match = VOWEL_SUFFIX.search(path.stem)
    return match.group(1).lower() if match else None

def speaker_from_filename(path):
    # Hapus token vowel paling akhir, termasuk nomor take (contoh G09_S01_A01 -> G09_S01).
    return VOWEL_SUFFIX.sub("", path.stem).rstrip("_- ") or path.stem

def vowel_from_relative_folders(path, base):
    # Cari komponen folder vowel dari yang paling dekat dengan file.
    for component in reversed(path.relative_to(base).parts[:-1]):
        if component.lower() in VOWELS:
            return component.lower()
    return None

def load_records(name, folder):
    records = []
    if not folder.is_dir():
        return records
    for path in sorted(folder.rglob("*")):
        if not path.is_file() or path.suffix.lower() != ".wav":
            continue
        sr, signal = wavfile.read(path)
        filename_vowel = vowel_from_filename(path)
        folder_vowel = vowel_from_relative_folders(path, folder)

        if name == "Open Speakers":
            # Direktori open/<vowel> adalah ground truth utama, bukan identitas speaker.
            vowel = folder_vowel or filename_vowel
            if folder_vowel and filename_vowel and folder_vowel != filename_vowel:
                print(f"PERINGATAN label folder/nama berbeda: {path} ({folder_vowel} vs {filename_vowel})")
            speaker = speaker_from_filename(path)
            speaker_group = speaker
        else:
            vowel = filename_vowel or folder_vowel
            relative_parts = path.relative_to(folder).parts
            # Folder tingkat pertama adalah nama speaker pada struktur template/closed.
            speaker_group = relative_parts[0] if len(relative_parts) > 1 else speaker_from_filename(path)
            speaker = speaker_group

        if vowel not in VOWELS:
            raise ValueError(f"Label vowel tidak dapat ditentukan untuk: {path}")
        records.append({
            "signal": signal,
            "sr": int(sr),
            "speaker": speaker,
            "speaker_group": speaker_group,
            "vowel": vowel,
            "path": path,
        })
    return records

raw = {name: load_records(name, folder) for name, folder in SCENARIOS.items()}

print("| Skenario | WAV | Grup/speaker | Distribusi vowel | Sample rate | Kanal | Tipe |")
print("|---|---:|---:|---|---|---|---|")
for name, records in raw.items():
    rates = sorted({r["sr"] for r in records})
    channels = sorted({1 if r["signal"].ndim == 1 else r["signal"].shape[1] for r in records})
    dtypes = sorted({str(r["signal"].dtype) for r in records})
    vowels = dict(sorted(Counter(r["vowel"] for r in records).items()))
    groups = len({r["speaker_group"] for r in records})
    print(f"| {name} | {len(records)} | {groups} | {vowels} | {rates} | {channels} | {dtypes} |")

train_raw = raw["Closed Recording"]
if not train_raw:
    raise FileNotFoundError(f"Tidak ada WAV data latih di {SCENARIOS['Closed Recording']}")
missing_train_vowels = set(VOWELS) - {r["vowel"] for r in train_raw}
if missing_train_vowels:
    raise ValueError(f"Data template tidak memiliki kelas vowel: {sorted(missing_train_vowels)}")


In [ ]:
def prepare_audio(signal, sr):
    # Konversi audio ke mono float dan target sample rate sebelum ekstraksi fitur.
    signal = np.asarray(signal)
    if signal.ndim == 2:
        signal = signal.astype(np.float64).mean(axis=1)
    elif signal.ndim == 1:
        signal = signal.astype(np.float64)
    else:
        raise ValueError(f"Bentuk audio tidak didukung: {signal.shape}")
    if not np.all(np.isfinite(signal)):
        raise ValueError("Sinyal audio mengandung NaN/Inf")
    if sr <= 0:
        raise ValueError(f"Sample rate tidak valid: {sr}")
    if sr != TARGET_SR:
        divisor = gcd(int(sr), TARGET_SR)
        signal = resample_poly(signal, TARGET_SR // divisor, int(sr) // divisor)
    if len(signal) < int(0.025 * TARGET_SR):
        raise ValueError(f"Audio terlalu pendek untuk satu frame MFCC ({len(signal)} sampel)")
    return signal

def add_features(records):
    prepared = []
    for record in records:
        signal = prepare_audio(record["signal"], record["sr"])
        prepared.append({**record, "feat": extract_mfcc(signal, TARGET_SR)})
    return prepared

# Ekstrak per path sekali; template digunakan lagi sebagai skenario closed-recording.
feature_cache = {}
for records in raw.values():
    for record in records:
        if record["path"] not in feature_cache:
            feature_cache[record["path"]] = add_features([record])[0]

train = [feature_cache[r["path"]] for r in train_raw]
test = {name: [feature_cache[r["path"]] for r in records] for name, records in raw.items()}
models = {
    "DTW": train_dtw(train),
    "Gaussian": train_gaussian(train),
}
predictors = {"DTW": predict_dtw, "Gaussian": predict_gaussian}

results = {}
for system, predict in predictors.items():
    for scenario, records in test.items():
        results[(system, scenario)] = [
            {
                "speaker": record["speaker"],
                "speaker_group": record["speaker_group"],
                "vowel": record["vowel"],
                "pred": predict(record["feat"], models[system]),
                "path": record["path"],
            }
            for record in records
        ]

def accuracy(rows):
    return 100 * sum(row["vowel"] == row["pred"] for row in rows) / len(rows) if rows else float("nan")

def md_table(headers, rows, percent=True):
    def fmt(value):
        if isinstance(value, (int, float, np.number)) and np.isfinite(value):
            return f"{value:.2f}%".replace(".", ",") if percent else str(int(value))
        return str(value)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines.extend("| " + " | ".join(fmt(value) for value in row) + " |" for row in rows)
    display(Markdown("\n".join(lines)))

systems = ["DTW", "Gaussian"]
scenarios = list(SCENARIOS)
summary = []
for system in systems:
    values = [accuracy(results[(system, scenario)]) for scenario in scenarios]
    available = [value for value in values if np.isfinite(value)]
    summary.append([system, *values, float(np.mean(available)) if available else float("nan")])
md_table(["Sistem", *scenarios, "Rata-rata skenario"], summary)


In [ ]:
# Akurasi per vowel untuk setiap sistem dan skenario.
vowel_rows = []
for vowel in VOWELS:
    values = [
        accuracy([row for row in results[(system, scenario)] if row["vowel"] == vowel])
        for system in systems for scenario in scenarios
    ]
    vowel_rows.append([vowel, *values])
vowel_rows.append([
    "Semua vowel",
    *[accuracy(results[(system, scenario)]) for system in systems for scenario in scenarios],
])
md_table(["Vowel aktual", *[f"{system} {scenario}" for system in systems for scenario in scenarios]], vowel_rows)

def speaker_table(selected, title):
    columns = [(system, scenario) for system in systems for scenario in selected]
    groups = sorted({
        row["speaker_group"]
        for system, scenario in columns
        for row in results[(system, scenario)]
    })
    rows = []
    for group in groups:
        values = [
            accuracy([row for row in results[(system, scenario)] if row["speaker_group"] == group])
            for system, scenario in columns
        ]
        valid = [value for value in values if np.isfinite(value)]
        rows.append([group, *values, float(np.mean(valid)) if valid else float("nan")])
    if rows:
        averages = [
            float(np.nanmean([row[index] for row in rows]))
            if any(np.isfinite(row[index]) for row in rows) else float("nan")
            for index in range(1, len(rows[0]))
        ]
        rows.append(["Rata-rata speaker", *averages])
    print(title)
    md_table(["Speaker/folder", *[f"{system} {scenario}" for system, scenario in columns], "Rata-rata"], rows)

# Pada template/closed, speaker_group berasal dari nama folder speaker.
speaker_table(["Closed Recording", "Closed Speakers"], "Akurasi per speaker — template dan closed")
# Pada open, grup analisis diambil dari prefix nama file; label prediksi tetap vowel.
speaker_table(["Open Speakers"], "Akurasi per speaker — open")


In [ ]:
# Confusion matrix: baris = vowel aktual, kolom = vowel prediksi.
CONFUSION_SCENARIO = "Open Speakers"
if not raw[CONFUSION_SCENARIO]:
    print(f"{CONFUSION_SCENARIO}: tidak ada data WAV; confusion matrix dilewati.")
else:
    for system in systems:
        rows = results[(system, CONFUSION_SCENARIO)]
        matrix = np.zeros((len(VOWELS), len(VOWELS)), dtype=int)
        for row in rows:
            matrix[VOWELS.index(row["vowel"]), VOWELS.index(row["pred"])] += 1
        print(f"\n{system} — {CONFUSION_SCENARIO}")
        md_table(
            ["Aktual \\ Prediksi", *[vowel.upper() for vowel in VOWELS]],
            [[vowel.upper(), *matrix[index].tolist()] for index, vowel in enumerate(VOWELS)],
            percent=False,
        )


## Analisis hasil

Gunakan tabel untuk membandingkan closed recordings, closed speakers, dan open. Closed recordings memakai data yang sama dengan template sehingga bukan estimasi generalisasi. Pada skenario open, ground truth vowel berasal dari folder (`open/A`, `open/E`, dan seterusnya); identitas speaker tidak digunakan sebagai kelas yang diprediksi. DTW mempertahankan perbandingan urutan temporal, sedangkan Gaussian merangkum distribusi frame tanpa urutan.

Jangan menyimpulkan gender, perangkat, atau kondisi ruangan tanpa catatan perekaman pendukung.


In [ ]:
# Ekspor prediksi agar hasil dapat ditinjau tanpa menjalankan ulang evaluasi.
out_dir = ROOT / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "predictions.csv"
fieldnames = ["system", "scenario", "speaker_group", "speaker", "actual", "predicted", "path"]

with out_path.open("w", newline="", encoding="utf-8") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    writer.writeheader()
    for (system, scenario), rows in results.items():
        for row in rows:
            writer.writerow({
                "system": system,
                "scenario": scenario,
                "speaker_group": row["speaker_group"],
                "speaker": row["speaker"],
                "actual": row["vowel"],
                "predicted": row["pred"],
                "path": row["path"].relative_to(ROOT).as_posix(),
            })

print(f"Prediksi disimpan: {out_path.relative_to(ROOT).as_posix()} ({sum(map(len, results.values()))} baris)")
